# 06 — Score, categorias de prioridade e Excel final
1. Normalização robusta 0–1 (truncada em p5–p95).
2. Sub-índices: **E** (exposição), **S** (sensibilidade), **C** (capacidade adaptativa).
3. `Score = wE·E + wS·S + wC·(1 − C)`
4. Categorias P1 (crítica) → P4 por quebras naturais (Jenks); conferência com k-means.
5. Robustez: Monte Carlo nos pesos → % de vezes em P1.
6. `outputs/ranking_escolas.xlsx` — pesos e limiares editáveis na aba **parametros** (score e categoria recalculam por fórmula).

In [ ]:
!pip -q install geopandas mapclassify scikit-learn openpyxl

In [ ]:
# --- setup comum (Colab) ---
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/2_PESQUISA/02_PROJETOS-PESQUISA-MODELAGEM/ESCOLAS-CONFORTO-TERMICO-SME'
import os, sys
os.chdir(ROOT); sys.path.append(f'{ROOT}/src')
RAW, INT, PRO, OUT = [f'{ROOT}/{p}' for p in ['data/raw', 'data/interim', 'data/processed', 'outputs']]
for p in [RAW, INT, PRO, OUT]:
    os.makedirs(p, exist_ok=True)
CRS = 'EPSG:31983'          # SIRGAS 2000 / UTM 23S
BUFFERS = [50, 100, 150, 250]
BUFFER_REF = 100            # buffer usado no score (≈ 1 pixel termal nativo do Landsat)

In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, mapclassify
from sklearn.cluster import KMeans
from indices import norm_robusta

esc = gpd.read_file(f'{INT}/escolas.gpkg', layer='escolas'); esc['cod_esc'] = esc['cod_esc'].astype(str)
temp = pd.read_parquet(f'{PRO}/temperatura_escolas.parquet')
ind = pd.read_parquet(f'{PRO}/indices_escolas.parquet')
vul = pd.read_parquet(f'{PRO}/vulnerabilidade_escolas.parquet')
for d in (temp, ind, vul): d['cod_esc'] = d['cod_esc'].astype(str)
df = (esc.drop(columns='geometry').merge(temp, on='cod_esc', how='left')
      .merge(ind, on='cod_esc', how='left').merge(vul.drop(columns=['NDVI_median'], errors='ignore'), on='cod_esc', how='left'))

PESOS = {'E': 0.5, 'S': 0.3, 'C': 0.2}
INDIC = {   # indicador: (sub-índice, inverter?)
    'GH26_ano': ('E', False), 'h_HI_ge32_ano': ('E', False), 'LST_p90': ('E', False), 'ANOM_mean': ('E', False),
    'peso_idade': ('S', False), 'integral': ('S', False), 'matriculas': ('S', False), 'renda_media': ('S', True),
    'NDVI_median': ('C', False), 'pct_vegetacao': ('C', False), 'possui_ac': ('C', False),
}
INDIC = {k: v for k, v in INDIC.items() if k in df and df[k].notna().any()}
print('indicadores usados:', list(INDIC))

In [ ]:
for k, (_g, inv) in INDIC.items():
    x = np.log1p(df[k]) if k == 'matriculas' else df[k]
    df[f'n_{k}'] = norm_robusta(x, inverter=inv).fillna(0.5)    # faltante → valor neutro
for g in 'ESC':
    cols = [f'n_{k}' for k, (gg, _) in INDIC.items() if gg == g]
    df[g] = df[cols].mean(axis=1) if cols else 0.0
df['score'] = PESOS['E'] * df['E'] + PESOS['S'] * df['S'] + PESOS['C'] * (1 - df['C'])

nb = mapclassify.NaturalBreaks(df['score'], k=4)
LIM = list(nb.bins[:-1])                     # 3 limiares
df['categoria'] = pd.cut(df['score'], [-np.inf] + LIM + [np.inf], labels=['P4', 'P3', 'P2', 'P1']).astype(str)
print(df['categoria'].value_counts().sort_index(), '\nlimiares Jenks:', np.round(LIM, 3))

km = KMeans(4, n_init=20, random_state=0).fit(df[['E', 'S', 'C']])
df['cluster_km'] = km.labels_
print(pd.crosstab(df['categoria'], df['cluster_km']))

In [ ]:
# Monte Carlo: pesos ~ Dirichlet em torno dos pesos-base; P1 = mesma fração de escolas do cenário base
rng = np.random.default_rng(42); N = 1000
w = rng.dirichlet(np.array(list(PESOS.values())) * 50, N)
frac_p1 = (df['categoria'] == 'P1').mean()
M = np.c_[df['E'], df['S'], 1 - df['C']]
sc = M @ w.T                                         # (escolas, N)
corte = np.quantile(sc, 1 - frac_p1, axis=0)
df['estab_P1_pct'] = ((sc >= corte).mean(axis=1) * 100).round(1)
df['rank'] = df['score'].rank(ascending=False, method='min').astype(int)
df.sort_values('rank')[['rank', 'nome', 'tipoesc', 'rede_tipo', 'dre', 'score', 'categoria', 'estab_P1_pct']].head(20)

In [ ]:
import matplotlib.pyplot as plt
g = esc.merge(df[['cod_esc', 'categoria']], on='cod_esc')
cores = {'P1': '#b2182b', 'P2': '#ef8a62', 'P3': '#fddbc7', 'P4': '#d1e5f0'}
ax = g.plot(color=g['categoria'].map(cores), markersize=3, figsize=(7, 9))
for k, c in cores.items(): ax.scatter([], [], c=c, label=k)
ax.legend(title='Prioridade'); ax.set_axis_off(); plt.title('Prioridade de climatização — escolas municipais')
plt.savefig(f'{OUT}/mapa_prioridade.png', dpi=200, bbox_inches='tight')

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter

wb = Workbook(); H = Font(bold=True, color='FFFFFF'); HF = PatternFill('solid', fgColor='1F4E78')

def aba(nome, d):
    ws = wb.create_sheet(nome); ws.append(list(d.columns))
    for row in d.itertuples(index=False):
        ws.append([None if (isinstance(v, float) and np.isnan(v)) else v for v in row])
    for c in ws[1]: c.font, c.fill = H, HF
    ws.freeze_panes = 'B2'; ws.auto_filter.ref = ws.dimensions
    for i, col in enumerate(d.columns, 1):
        ws.column_dimensions[get_column_letter(i)].width = min(max(len(str(col)) + 2, 10), 40)
    return ws

leia = wb.active; leia.title = 'leia-me'
for l in ['Priorização de climatização — escolas municipais de SP (direta + conveniada; Infantil e Fundamental)',
          f'Escolas: {len(df)} | buffer de referência: {BUFFER_REF} m | LST Landsat 8/9 (meses letivos quentes)',
          'Score = wE·E + wS·S + wC·(1−C); pesos e limiares na aba parametros (editáveis).',
          'Abas: escolas, temperatura, indices, vulnerabilidade, score (fórmulas), ranking (valores), parametros, dicionario.',
          'Metodologia completa: docs/metodologia.md']:
    leia.append([l])
leia.column_dimensions['A'].width = 120

idc = ['cod_esc', 'nome']
aba('escolas', df[[c for c in ['cod_esc', 'cod_inep', 'nome', 'tipoesc', 'etapa', 'rede_tipo', 'dre', 'distrito',
                               'subpref', 'endereco', 'lat', 'lon', 'matriculas'] if c in df]])
aba('temperatura', df[idc + [c for c in temp.columns if c != 'cod_esc']].round(2))
aba('indices', df[idc + [c for c in ind.columns if c != 'cod_esc']].round(2))
aba('vulnerabilidade', df[idc + [c for c in vul.columns if c != 'cod_esc' and c in df.columns]].round(3))

# parametros
par = wb.create_sheet('parametros')
par.append(['parametro', 'valor', 'descricao'])
par.append(['peso_E', PESOS['E'], 'peso da exposição'])
par.append(['peso_S', PESOS['S'], 'peso da sensibilidade'])
par.append(['peso_C', PESOS['C'], 'peso da capacidade adaptativa (entra como 1−C)'])
par.append(['lim_P3', LIM[0], 'score ≥ → P3'])
par.append(['lim_P2', LIM[1], 'score ≥ → P2'])
par.append(['lim_P1', LIM[2], 'score ≥ → P1'])
for c in par[1]: c.font, c.fill = H, HF
par.column_dimensions['A'].width = 12; par.column_dimensions['C'].width = 45

# score com fórmulas
ncols = [f'n_{k}' for k in INDIC]
sc_df = df[idc + ncols + ['E', 'S', 'C']].round(4)
ws = aba('score', sc_df)
ce, cs, cc = [get_column_letter(sc_df.columns.get_loc(x) + 1) for x in ('E', 'S', 'C')]
k_sc = sc_df.shape[1] + 1; k_cat = k_sc + 1
ws.cell(1, k_sc, 'score').font = H; ws.cell(1, k_sc).fill = HF
ws.cell(1, k_cat, 'categoria').font = H; ws.cell(1, k_cat).fill = HF
L = get_column_letter(k_sc)
for r in range(2, len(sc_df) + 2):
    ws.cell(r, k_sc, f'=parametros!$B$2*{ce}{r}+parametros!$B$3*{cs}{r}+parametros!$B$4*(1-{cc}{r})').number_format = '0.000'
    ws.cell(r, k_cat, f'=IF({L}{r}>=parametros!$B$7,"P1",IF({L}{r}>=parametros!$B$6,"P2",IF({L}{r}>=parametros!$B$5,"P3","P4")))')
ws.auto_filter.ref = ws.dimensions

# ranking (valores do cenário base)
rk = df.sort_values('rank')[['rank', 'cod_esc', 'nome', 'tipoesc', 'etapa', 'rede_tipo', 'dre', 'score', 'categoria',
                             'estab_P1_pct', 'Tar_media_aula', 'GH26_ano', 'h_HI_ge32_ano', 'LST_p90']].round(3)
wr = aba('ranking', rk)
fill = {k: PatternFill('solid', fgColor=v.strip('#')) for k, v in cores.items()}
ci = rk.columns.get_loc('categoria') + 1
for r in range(2, len(rk) + 2):
    wr.cell(r, ci).fill = fill.get(wr.cell(r, ci).value, PatternFill())

DIC = {
 'LST_*': 'temperatura de superfície Landsat (°C) — média ponderada por fração de pixel no buffer; sufixo = estatística entre cenas',
 'ANOM_*': 'LST da escola − LST média da área na mesma cena (°C) = intensidade da ilha de calor de superfície',
 'LST_t_*': 'estatísticas da série cena a cena (variação temporal); amplitude = máx − mín',
 'NDVI_median': 'índice de vegetação mediano no buffer',
 'dT_ar_estimado': 'diferença de temperatura do ar da escola em relação às estações INMET (°C) = β·Δanomalia',
 'Tar_*_aula': 'temperatura do ar estimada em horas letivas (°C)',
 'HI_*': 'Heat Index / sensação térmica NOAA (°C)', 'DI_*': 'Índice de Desconforto de Thom (°C)',
 'GH26_ano': 'graus-hora acima de 26 °C em horas letivas, média anual',
 'h_HI_ge32_ano': 'horas letivas/ano com sensação térmica ≥ 32 °C (cautela extrema)',
 'h_DI_ge24_ano': 'horas letivas/ano com DI ≥ 24 (>50% da população desconfortável)',
 'peso_idade': 'peso pela faixa etária atendida (CEI=1 … EMEF=0,5)', 'integral': 'atendimento em período integral (1/0)',
 'n_*': 'indicador normalizado 0–1 (p5–p95)', 'E,S,C': 'sub-índices de exposição, sensibilidade e capacidade adaptativa',
 'score': 'wE·E + wS·S + wC·(1−C)', 'categoria': 'P1 (crítica) … P4, por quebras naturais (Jenks)',
 'estab_P1_pct': '% de 1000 simulações de pesos em que a escola fica em P1',
}
aba('dicionario', pd.DataFrame(DIC.items(), columns=['campo', 'descricao']))
wb.save(f'{OUT}/ranking_escolas.xlsx'); print('salvo', f'{OUT}/ranking_escolas.xlsx')